In [1]:
# Descarga del dataset desde Kaggle usando kagglehub
import kagglehub
import os

# Con este método siempre se tiene el último dataset disponible
descarga_dir = kagglehub.dataset_download("msalman97/dataset-for-traffic-sign-master-app")

# Vamos a acceder a la carpeta descomprimida "Dataset"
dataset_dir = os.path.join(descarga_dir, "Dataset")
print("Directorio de descarga:", dataset_dir)

/home/christianr/umia_projects/UMU-MAI/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Directorio de descarga: /home/christianr/.cache/kagglehub/datasets/msalman97/dataset-for-traffic-sign-master-app/versions/1/Dataset


In [2]:
# Este código modifica la función existente para usar ultralytics en lugar del script personalizado.
# Asegúrate de tener instalada la librería ultralytics: pip install ultralytics

import os
import shutil
import random
from pathlib import Path
import pandas as pd
from PIL import Image
import yaml
from ultralytics import YOLO

"""
Flujo para preparar datos y lanzar el fine-tuning de un modelo YOLOv11 usando ultralytics.
- train_dir: directorio raíz que contiene las imágenes (ej. carpeta padre de "Train").
- labels_csv: CSV con las anotaciones en formato: Width,Height,Roi.X1,Roi.Y1,Roi.X2,Roi.Y2,ClassId,Path
  (Width/Height son del tamaño de la imagen; Roi.X1/Y1/X2/Y2 son xmin/ymin/xmax/ymax absolutos; ClassId es numérico; Path es ruta relativa).
Salida:
- Crea una estructura preparada en ./dataset_prepared/{train,val}/images & labels
- Genera data.yaml compatible con ultralytics
- Llama a model.train() de ultralytics con los parámetros indicados
"""

def prepare_and_finetune_yolo11_ultralytics(train_dir: str,
                                           labels_csv: str,
                                           model_path: str = "models/yolov11n.pt",
                                           out_dir: str = "dataset_prepared",
                                           val_split: float = 0.1,
                                           epochs: int = 50,
                                           batch_size: int = 16,
                                           img_size: int = 640,
                                           seed: int = 42):
    random.seed(seed)
    train_dir = Path(train_dir)
    labels_csv = Path(labels_csv)
    out_dir = Path(out_dir)
    
    # Leer CSV y mapear columnas
    df = pd.read_csv(labels_csv)
    df.columns = [c.strip() for c in df.columns]  # Normalizar nombres
    required_cols = ["Width", "Height", "Roi.X1", "Roi.Y1", "Roi.X2", "Roi.Y2", "ClassId", "Path"]
    if not all(c in df.columns for c in required_cols):
        raise ValueError(f"CSV debe contener columnas: {required_cols}")
    
    # Renombrar para consistencia
    df = df.rename(columns={
        "Roi.X1": "xmin", "Roi.Y1": "ymin", "Roi.X2": "xmax", "Roi.Y2": "ymax",
        "ClassId": "class_id", "Path": "filename"
    })
    df["class_id"] = df["class_id"].astype(int)
    
    # Obtener clases únicas
    classes = sorted(df["class_id"].unique().astype(str).tolist())
    
    # Crear estructura de salida
    for split in ("train", "val"):
        (out_dir / split / "images").mkdir(parents=True, exist_ok=True)
        (out_dir / split / "labels").mkdir(parents=True, exist_ok=True)
    
    # Encontrar imágenes disponibles
    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    filenames = sorted(df["filename"].unique().tolist())
    available = {}
    for p in train_dir.rglob("*"):
        if p.suffix.lower() in img_exts:
            available[p.name] = p  # Asumir que filename es el nombre del archivo
    image_files = [f for f in filenames if Path(f).name in available]
    if not image_files:
        raise FileNotFoundError("No se encontraron imágenes referenciadas en el CSV dentro de train_dir")
    
    # Split train/val
    random.shuffle(image_files)
    val_count = max(1, int(len(image_files) * val_split))
    val_imgs = set(image_files[:val_count])
    train_imgs = set(image_files[val_count:])
    
    # Función para convertir bbox a YOLO normalizado
    def xyxy_to_yolo(xmin, ymin, xmax, ymax, iw, ih):
        x_center = ((xmin + xmax) / 2.0) / iw
        y_center = ((ymin + ymax) / 2.0) / ih
        w = (xmax - xmin) / iw
        h = (ymax - ymin) / ih
        return x_center, y_center, w, h
    
    # Procesar imágenes y generar labels
    for name in image_files:
        p_src = available[Path(name).name]
        with Image.open(p_src) as im:
            iw, ih = im.size
        split = "val" if name in val_imgs else "train"
        dst_img = out_dir / split / "images" / Path(name).name
        shutil.copy2(p_src, dst_img)
        
        rows = df[df["filename"] == name]
        label_lines = []
        for _, r in rows.iterrows():
            cls_id = int(r["class_id"])
            xmin = float(r["xmin"])
            ymin = float(r["ymin"])
            xmax = float(r["xmax"])
            ymax = float(r["ymax"])
            x_c, y_c, w, h = xyxy_to_yolo(xmin, ymin, xmax, ymax, iw, ih)
            label_lines.append(f"{cls_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")
        
        dst_lbl = out_dir / split / "labels" / (Path(name).stem + ".txt")
        with open(dst_lbl, "w", encoding="utf8") as fh:
            fh.write("\n".join(label_lines))
    
    # Generar data.yaml
    data_yaml = {
        "train": str((out_dir / "train" / "images").resolve()),
        "val": str((out_dir / "val" / "images").resolve()),
        "nc": len(classes),
        "names": classes
    }
    data_yaml_path = out_dir / "data.yaml"
    with open(data_yaml_path, "w", encoding="utf8") as fh:
        yaml.safe_dump(data_yaml, fh, sort_keys=False)
    
    # Cargar modelo y entrenar con ultralytics
    model = YOLO(model_path)
    model.train(
        data=str(data_yaml_path),
        epochs=epochs,
        batch=batch_size,
        imgsz=img_size,
        seed=seed
    )
    
    print("Fine-tuning completado. Resultados en el directorio runs/detect/train (por defecto de ultralytics).")
    return {"prepared_dataset": str(out_dir.resolve()), "data_yaml": str(data_yaml_path.resolve())}



In [3]:
training_dir = os.path.join(dataset_dir, "Train")
etiquetas = os.path.join(dataset_dir, "Train.csv")

In [ ]:
# Ejemplo de uso:
prepare_and_finetune_yolo11_ultralytics(
    train_dir=training_dir,  # Carpeta que contiene "Train"
    labels_csv=etiquetas,
    model_path="models/yolo11n.pt",
    epochs=10,
    batch_size=40,
    img_size=640
)

Ultralytics 8.3.221 🚀 Python-3.12.3 torch-2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=40, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_prepared/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=models/yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, per